In [1]:
import pandas as pd
from pathlib import Path
git_root = Path.cwd().resolve().parents[2]

## Step 1 
- Get All Demand Profiles and Harmonize timestamp index

In [2]:
demand_profiles_root = Path(git_root/'data/processed/SpecifiedDemandProfile_representative_years/Specified_Demand_Profile')
countries = ['BA', 'RS', 'XK']
SDP = {}
for country in countries:
    country_profile_path = demand_profiles_root / f'euclidean_representative_profile_{country}.csv'
    df = pd.read_csv(country_profile_path)
    year = df.columns[0]
    date_range = pd.date_range(start=f'{year}-01-01 01:00', periods=8760, freq='h')
    # Ensure df has the correct number of rows by reindexing
    df = df.set_index(pd.RangeIndex(len(df)))
    df = df.reindex(range(8760))
    df.index = date_range
    SDP[country] = df

## Step 2
- Create a Combined Dataframe with All country Profiles in OSeMOSYS Specified Demand Profile Schema

In [3]:
import osemosys_clusterting as osms_clustering

osemosys_sdp_dfs = {}
for country in countries:
    osemosys_sdp_dfs[country] = osms_clustering.generate_demand_profile(
        fuels=['ELC'],
        df=SDP[country],
        region=country,
        base_year=2020
    )

# Step 3:

- Extract Representative Days(decide what type of normalization method you want to pick.) 
- Check the README file in this folder reagarding tips for selection on profile normalization methods.

In [4]:
# Get representative days for all countries and store in a dict
representative_days_dict = {}
for country in countries:
    # Drop rows with NaN in VALUE column
    clean_profile = osemosys_sdp_dfs[country][['VALUE']]
    profiles = {
        'demand': clean_profile
    }
    print(f'Processing {country}...')
    representative_days_dict[country] = osms_clustering.get_representative_days_from_demand_profiles(
        profiles=profiles,
        n_clusters=5,
        normalization_method='euclidean', #'minmax
        profile_key_for_plot='demand',
        plot_save_to=git_root/f'vis/Specified_Demand_Profile/{country}',
        see=False # to see the plots
    )

Processing BA...
Processing RS...
Processing XK...


In [5]:
timeslices_dict = {}
for country in countries:
    print(f'Processing {country}...')
    func_args = {
        "osemosys_sdp_df": osemosys_sdp_dfs[country],
        "representative_days": representative_days_dict[country],
        "hour_grouping": 4,
        'profile_normalization_type': 'euclidean', # or 'minmax'
        "operation": "sum"
    }
    timeslices_dict[country] = osms_clustering.get_timeslices(**func_args)

Processing BA...
Processing Timeslices for 5 representative days with hour grouping of 4 hours (i.e., 6.0 groups per day)
Total timeslices to be constructed: 30
Processing RS...
Processing Timeslices for 5 representative days with hour grouping of 4 hours (i.e., 6.0 groups per day)
Total timeslices to be constructed: 30
Processing XK...
Processing Timeslices for 5 representative days with hour grouping of 4 hours (i.e., 6.0 groups per day)
Total timeslices to be constructed: 30


In [6]:
timeslices_all= pd.concat(timeslices_dict.values())
save_to_path = Path(git_root/'data/processed/osemosys')
save_to_path.mkdir(parents=True, exist_ok=True)
# Save the timeslices to a CSV file
timeslices_all.to_csv(save_to_path/'SpecifiedDemandProfile.csv', index=False)
print(f'Timeslices saved to: {save_to_path/"SpecifiedDemandProfile.csv"}')

Timeslices saved to: /local-scratch/localhome/mei3/eliasinul/work/WB-OEMC/data/processed/osemosys/SpecifiedDemandProfile.csv
